# Setup

Import Semantic Kernel SDK from pypi.org

In [1]:
# Note: if using a Poetry virtual environment, do not run this cell
%pip install semantic-kernel==1.2.0

Initial configuration for the notebook to run properly.

In [2]:
# Make sure paths are correct for the imports

import os
import sys

notebook_dir = os.path.abspath("")
parent_dir = os.path.dirname(notebook_dir)
grandparent_dir = os.path.dirname(parent_dir)


sys.path.append(grandparent_dir)

### Configuring the Kernel

Let's get started with the necessary configuration to run Semantic Kernel. For Notebooks, we require a `.env` file with the proper settings for the model you use. Create a new file named `.env` and place it in this directory. Copy the contents of the `.env.example` file from this directory and paste it into the `.env` file that you just created.

**NOTE: Please make sure to include `GLOBAL_LLM_SERVICE` set to either OpenAI, AzureOpenAI, or HuggingFace in your .env file. If this setting is not included, the Service will default to AzureOpenAI.**

#### Option 1: using OpenAI

Add your [OpenAI Key](https://openai.com/product/) key to your `.env` file (org Id only if you have multiple orgs):

```
GLOBAL_LLM_SERVICE="OpenAI"
OPENAI_API_KEY="sk-..."
OPENAI_ORG_ID=""
OPENAI_CHAT_MODEL_ID=""
OPENAI_TEXT_MODEL_ID=""
OPENAI_EMBEDDING_MODEL_ID=""
```
The names should match the names used in the `.env` file, as shown above.

#### Option 2: using Azure OpenAI

Add your [Azure Open AI Service key](https://learn.microsoft.com/azure/cognitive-services/openai/quickstart?pivots=programming-language-studio) settings to the `.env` file in the same folder:

```
GLOBAL_LLM_SERVICE="AzureOpenAI"
AZURE_OPENAI_API_KEY="..."
AZURE_OPENAI_ENDPOINT="https://..."
AZURE_OPENAI_CHAT_DEPLOYMENT_NAME="..."
AZURE_OPENAI_TEXT_DEPLOYMENT_NAME="..."
AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME="..."
AZURE_OPENAI_API_VERSION="..."
```
The names should match the names used in the `.env` file, as shown above.

For more advanced configuration, please follow the steps outlined in the [setup guide](./CONFIGURING_THE_KERNEL.md).

Let's define our kernel for this example.

In [3]:
from semantic_kernel import Kernel

kernel = Kernel()

We will load our settings and get the LLM service to use for the notebook.

In [4]:
from services import Service

from samples.service_settings import ServiceSettings

service_settings = ServiceSettings()

# Select a service to use for this notebook (available services: OpenAI, AzureOpenAI, HuggingFace)
selectedService = (
    Service.AzureOpenAI
    if service_settings.global_llm_service is None
    else Service(service_settings.global_llm_service.lower())
)
print(f"Using service type: {selectedService}")

Using service type: Service.AzureOpenAI


We now configure our Chat Completion service on the kernel.

In [5]:
# Remove all services so that this cell can be re-run without restarting the kernel
kernel.remove_all_services()

service_id = None
if selectedService == Service.OpenAI:
    from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion

    service_id = "default"
    kernel.add_service(
        OpenAIChatCompletion(
            service_id=service_id,
        ),
    )
elif selectedService == Service.AzureOpenAI:
    from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion

    service_id = "default"
    kernel.add_service(
        AzureChatCompletion(
            service_id=service_id,
        ),
    )

# Run a Semantic Function

**Step 3**: Load a Plugin and run a semantic function:


In [6]:
plugin = kernel.add_plugin(parent_directory="../../../prompt_template_samples/", plugin_name="ChildrensBookPlugin")

In [8]:
from semantic_kernel.functions import KernelArguments

book_function = plugin["CreateBook"]

book = await kernel.invoke(
    book_function,
    KernelArguments(INPUT="time travel to dinosaur age", numPages=7, numWordsPerPage=300),
)
print(book)

```json
[
  {
    "page": 1,
    "content": "Once upon a time, in a small town, there lived a curious girl named Lily and her adventurous little brother, Max. One sunny afternoon, they discovered a mysterious, glowing portal in their backyard. 'What do you think it is?' asked Max, eyes wide with excitement. 'I don't know, but let's find out!' replied Lily, grabbing her brother's hand. Together, they stepped through the portal and found themselves in a lush, prehistoric jungle filled with towering trees and strange sounds."
  },
  {
    "page": 2,
    "content": "The air was thick with the scent of wildflowers and the distant roars of unknown creatures. 'Wow, look at those giant ferns!' said Max, pointing to plants taller than their house. As they ventured deeper into the jungle, they stumbled upon a herd of gentle, long-necked dinosaurs munching on leaves. 'Those are Brachiosaurus,' whispered Lily, remembering a book she had read. The dinosaurs glanced at the children but continued the